In [2]:
import sys
sys.path.insert(0, "/home/as5606/projects/LLaVA-Med")

from llava.model.builder import load_pretrained_model
from llava.mm_utils import get_model_name_from_path

MODEL_PATH = "/home/as5606/projects/LLaVA-Med/models/llava-med-v1.5-mistral-7b"

model_name = get_model_name_from_path(MODEL_PATH)
tokenizer, model, image_processor, context_len = load_pretrained_model(
    MODEL_PATH, None, model_name, load_4bit=False, load_8bit=False, device_map="auto"
)
model.eval()

print("Model loaded:", model_name)
print("Context length:", context_len)

/home/as5606/miniconda3/envs/llava-train/lib/python3.10/site-packages/torch/cuda/__init__.py:51: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/as5606/miniconda3/envs/llava-train/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/as5606/miniconda3/envs/llava-train/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|█████████████████████████████████

Model loaded: llava-med-v1.5-mistral-7b
Context length: 2048


In [3]:
import torch
from PIL import Image
from llava.mm_utils import process_images, tokenizer_image_token
from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN
from llava.conversation import conv_templates

In [4]:
def run_inference(question, image_path, conv_mode="llava_v1"):
    prompt = DEFAULT_IMAGE_TOKEN + "\n" + question

    conv = conv_templates[conv_mode].copy()
    conv.append_message(conv.roles[0], prompt)
    conv.append_message(conv.roles[1], None)
    full_prompt = conv.get_prompt()

    input_ids = tokenizer_image_token(
        full_prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt"
    ).unsqueeze(0).cuda()

    image = Image.open(image_path).convert("RGB")
    images_tensor = process_images([image], image_processor, model.config).to(
        model.device, dtype=torch.float16
    )

    with torch.inference_mode():
        output_ids = model.generate(
            input_ids,
            images=images_tensor,
            image_sizes=[image.size],
            do_sample=False,
            max_new_tokens=512,
            use_cache=True,
        )

    return tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0].strip()

In [5]:
import json

VAL_FILE = "/home/as5606/projects/Super_LlaVA_Med/dataset/x_ray/open_ended_train_and_val/val_open_ended_annotations.jsonl"

with open(VAL_FILE) as f:
    sample = json.loads(f.readline())

print("Question :", sample["question"])
print("Type     :", sample["type"])
print("Gold     :", sample["answer"])
print("Image    :", sample["image_path"])
print()

pred = run_inference(sample["question"], sample["image_path"])
print("Prediction:", pred)

Question : What do the green arrows indicate?
Type     : standard
Gold     : The green arrows in the image are pointing to specific findings in the CT scan of the thorax. These findings are likely related to the abnormalities or changes in the patient's chest that were previously observed in a chest X-ray. The CT scan serves to confirm those findings.
Image    : /shared/rc/vlm-idk/LLaVA/data/images/25688319_fig2.jpg



The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Prediction: The green arrows in the CT images are pointing to the tumor.


In [8]:
import json
from tqdm import tqdm
import warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

VAL_FILE = "/home/as5606/projects/Super_LlaVA_Med/dataset/x_ray/open_ended_train_and_val/val_open_ended_annotations.jsonl"
OUT_FILE  = "/home/as5606/projects/Super_LlaVA_Med/model_accuracy/xray_predictions.jsonl"

records = [json.loads(l) for l in open(VAL_FILE)]

with open(OUT_FILE, "w") as out:
    for rec in tqdm(records, desc="Evaluating"):
        pred = run_inference(rec["question"], rec["image_path"])
        out.write(json.dumps({
            "index":    rec["index"],
            "type":     rec["type"],
            "question": rec["question"],
            "gold":     rec["answer"],
            "pred":     pred,
        }) + "\n")
        out.flush()

print(f"Saved to {OUT_FILE}")


Evaluating:   0%|                                                                                                                                    | 0/858 [00:00<?, ?it/s]

Evaluating: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 858/858 [15:55<00:00,  1.11s/it]

Saved to /home/as5606/projects/Super_LlaVA_Med/model_accuracy/xray_predictions.jsonl


In [9]:
import json
from tqdm import tqdm
import warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

VAL_FILE = "/home/as5606/projects/Super_LlaVA_Med/dataset/MRI/open_ended_train_and_val/val_open_ended_annotations.jsonl"
OUT_FILE  = "/home/as5606/projects/Super_LlaVA_Med/model_accuracy/mri_predictions.jsonl"


records = [json.loads(l) for l in open(VAL_FILE)]

with open(OUT_FILE, "w") as out:
    for rec in tqdm(records, desc="Evaluating"):
        pred = run_inference(rec["question"], rec["image_path"])
        out.write(json.dumps({
            "index":    rec["index"],
            "type":     rec["type"],
            "question": rec["question"],
            "gold":     rec["answer"],
            "pred":     pred,
        }) + "\n")
        out.flush()

print(f"Saved to {OUT_FILE}")


Evaluating: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2936/2936 [54:41<00:00,  1.12s/it]

Saved to /home/as5606/projects/Super_LlaVA_Med/model_accuracy/mri_predictions.jsonl


In [10]:
import json
from tqdm import tqdm
import warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

VAL_FILE = "/home/as5606/projects/Super_LlaVA_Med/dataset/CT/open_ended_train_and_val/val_open_ended_annotations.jsonl"
OUT_FILE  = "/home/as5606/projects/Super_LlaVA_Med/model_accuracy/ct_predictions.jsonl"



records = [json.loads(l) for l in open(VAL_FILE)]

with open(OUT_FILE, "w") as out:
    for rec in tqdm(records, desc="Evaluating"):
        pred = run_inference(rec["question"], rec["image_path"])
        out.write(json.dumps({
            "index":    rec["index"],
            "type":     rec["type"],
            "question": rec["question"],
            "gold":     rec["answer"],
            "pred":     pred,
        }) + "\n")
        out.flush()

print(f"Saved to {OUT_FILE}")


Evaluating: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3660/3660 [59:25<00:00,  1.03it/s]

Saved to /home/as5606/projects/Super_LlaVA_Med/model_accuracy/ct_predictions.jsonl


In [7]:
import json
from tqdm import tqdm
import warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

VAL_FILE = "/home/as5606/projects/Super_LlaVA_Med/dataset/histology/open_ended_train_and_val/val_open_ended_annotations.jsonl"
OUT_FILE  = "/home/as5606/projects/Super_LlaVA_Med/model_accuracy/histology_predictions.jsonl"




records = [json.loads(l) for l in open(VAL_FILE)]

# find already-processed indices
done = set()
import os
if os.path.exists(OUT_FILE):
    with open(OUT_FILE) as f:
        for line in f:
            done.add(json.loads(line)["index"])

with open(OUT_FILE, "a") as out:  # append, not write
    for rec in tqdm(records, desc="Evaluating"):
        if rec["index"] in done:
            continue
        pred = run_inference(rec["question"], rec["image_path"])
        out.write(json.dumps({
            "index":    rec["index"],
            "type":     rec["type"],
            "question": rec["question"],
            "gold":     rec["answer"],
            "pred":     pred,
        }) + "\n")
        out.flush()



Evaluating: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7550/7550 [01:25<00:00, 88.61it/s]


In [8]:
import json
from tqdm import tqdm
import warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

VAL_FILE = "/home/as5606/projects/Super_LlaVA_Med/dataset/gross_pathology/open_ended_train_and_val/val_open_ended_annotations.jsonl"
OUT_FILE  = "/home/as5606/projects/Super_LlaVA_Med/model_accuracy/gross_pathology_predictions.jsonl"





records = [json.loads(l) for l in open(VAL_FILE)]

with open(OUT_FILE, "w") as out:
    for rec in tqdm(records, desc="Evaluating"):
        pred = run_inference(rec["question"], rec["image_path"])
        out.write(json.dumps({
            "index":    rec["index"],
            "type":     rec["type"],
            "question": rec["question"],
            "gold":     rec["answer"],
            "pred":     pred,
        }) + "\n")
        out.flush()

print(f"Saved to {OUT_FILE}")


Evaluating: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 968/968 [13:45<00:00,  1.17it/s]

Saved to /home/as5606/projects/Super_LlaVA_Med/model_accuracy/gross_pathology_predictions.jsonl
